# 02 Point Forecast Baselines

这个 notebook 整理最终 5 条月度点预测基线：Baseline 1、Baseline 2、Baseline 3、Baseline 4 和 AK-D。公开版使用合成样例数据跑通流程，并用脱敏真实指标做结果分析。

## 1. Data Description

输入包括月度真值、气象预报、功率曲线和 AK-D 分月订正系数。真实项目中的 EC45、ERA5、SCADA 和月度理论电量不会上传；公开样例只保留字段结构。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from wind_power_baselines.baselines import build_all_baselines
from wind_power_baselines.evaluation import evaluate_predictions, normalize_public_predictions
from wind_power_baselines.preprocessing import clean_monthly_truth, prepare_forecast

sns.set_theme(style='whitegrid')
truth = pd.read_csv(PROJECT_ROOT / 'data_sample' / 'sample_monthly_truth.csv')
forecast_raw = pd.read_csv(PROJECT_ROOT / 'data_sample' / 'sample_weather_forecast.csv')
power_curve = pd.read_csv(PROJECT_ROOT / 'data_sample' / 'sample_power_curve.csv')
correction = pd.read_csv(PROJECT_ROOT / 'data_sample' / 'sample_ak_monthly_correction.csv')
truth.head(), forecast_raw.head(), power_curve.head(), correction.head()

## 2. Cleaning And Preprocessing

这里完成月份标准化、时间解析、风速合法性检查、预测提前期检查和预测月份对齐。公开样例故意保持简单，重点是展示真实工程里需要固定的数据契约。

In [ ]:
truth_clean = clean_monthly_truth(truth)
forecast = prepare_forecast(forecast_raw)
print({'truth_months': len(truth_clean), 'forecast_rows': len(forecast), 'forecast_months': forecast['month'].nunique()})
forecast.head()

## 3. EDA

EDA 先回答三个问题：真值有没有季节性，预报风速覆盖哪些月份，功率曲线是否能覆盖样例风速范围。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.lineplot(data=truth_clean, x='month', y='energy_true', marker='o', ax=axes[0])
sns.boxplot(data=forecast, x='month', y='wind_speed', ax=axes[1])
sns.lineplot(data=power_curve, x='wind_speed_bin', y='power_kw', hue='season', estimator='mean', errorbar=None, ax=axes[2])
axes[0].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='x', rotation=45)
axes[0].set_title('Monthly truth sample')
axes[1].set_title('Forecast wind speed by month')
axes[2].set_title('Power curve coverage')
fig.tight_layout()

## 4. Baseline Methods

- Baseline 1：历史同期电量均值，作为无气象信息的朴素基线。
- Baseline 2：ERA5/气候库思想，先用功率曲线得到历史电量库，再取 P50。
- Baseline 3：EC45 + LLS 订正 + 功率曲线，强调中长期预报风速到电量的映射。
- Baseline 4：EC45 + 相似日/覆盖优选 + 功率曲线，强调从历史相似状态借用电量水平。
- AK-D：分月订正 AK 方法，把订正系数按月份拆开，用来缓解全年同一订正系数导致的小风月高估和大风月低估。

In [ ]:
predictions = build_all_baselines(truth_clean, forecast, power_curve, correction)
predictions

## 5. Evaluation

统一计算 MAPE、汇总偏差和 PearsonR。真实评估中还会按 W1/W2/W3 等窗口拆分；公开样例只演示接口，真实结果在 `results_public` 中以脱敏形式给出。

In [ ]:
metrics = evaluate_predictions(predictions)
public_predictions = normalize_public_predictions(predictions)
metrics, public_predictions.head()

## 6. Public Result Analysis

下面读取真实工程脱敏后的结果。这里不展示真实 kWh/GWh，只展示每个方法在自身可用月份上的 MAPE、Bias、PearsonR，以及逐月归一化指数。

In [ ]:
public_metrics = pd.read_csv(PROJECT_ROOT / 'results_public' / 'public_metrics_summary.csv')
public_monthly = pd.read_csv(PROJECT_ROOT / 'results_public' / 'public_monthly_predictions_index.csv')
public_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=public_metrics, x='method', y='mape_percent', ax=axes[0])
sns.barplot(data=public_metrics, x='method', y='bias_percent', ax=axes[1])
for axis in axes:
    axis.tick_params(axis='x', rotation=30)
axes[0].set_title('Public MAPE by method')
axes[1].set_title('Public bias by method')
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=public_monthly, x='month', y='energy_pred_index', hue='method', marker='o', ax=ax)
sns.lineplot(data=public_monthly.drop_duplicates('month'), x='month', y='energy_true_index', color='black', marker='o', label='Truth index', ax=ax)
ax.tick_params(axis='x', rotation=45)
ax.set_title('Normalized monthly predictions and truth')
ax.set_ylabel('Index, truth-window mean = 1')
fig.tight_layout()

## 7. Conclusion

从真实脱敏结果看，5 条最终点预测基线覆盖了从朴素历史同期、气候库功率曲线、EC45 订正、相似日，到 AK-D 分月订正的完整路线。AK-D 的价值是说明订正系数按月份拆分后，部分极端月份误差会回落；它仍需要关注分月样本少导致的过修正。